In [1]:
#%matplotlib widget
import numpy as np
import matplotlib, matplotlib.pyplot as plt

from dr_utils import *
from error_calc import *

## Data Loading

In [2]:
from scipy.io import loadmat

In [3]:
Dataset = loadmat("sensor_data_u.mat")
print(Dataset.keys())

dict_keys(['__header__', '__version__', '__globals__', 'time_grid', 'high_end_imu', 'low_end_imu_I', 'low_end_imu_ID', 'odometer', 'reference_ins'])


In [4]:
time_grid = Dataset['time_grid'].flatten()
print(time_grid.shape)

(2496,)


In [5]:
print(Dataset['high_end_imu'].dtype.names)
novatel_acc_data = Dataset['high_end_imu']['accelerometer'][0,0]
print(novatel_acc_data.shape)
novatel_gyro_data = Dataset['high_end_imu']['gyroscope'][0,0].flatten()
print(novatel_gyro_data.shape)

# Unpack the nested arrays and extract the bias values
# novatel_biases = {
#     "bx": Dataset['high_end_imu']['biases'][0][0][0][0][0].item(),
#     "by": Dataset['high_end_imu']['biases'][0][0][0][0][1].item(),
#     "bz": Dataset['high_end_imu']['biases'][0][0][0][0][2].item(),
#     "bw": Dataset['high_end_imu']['biases'][0][0][0][0][3].item()
# }

# Unpack the nested arrays and extract the bias values
novatel_biases = {
    "bx": 0,
    "by": 0,
    "bz": 0,
    "bw": 0
}

print(novatel_biases.items())

('accelerometer', 'gyroscope', 'biases')
(2496, 3)
(2496,)
dict_items([('bx', 0), ('by', 0), ('bz', 0), ('bw', 0)])


In [ ]:
print(Dataset['low_end_imu_I'].dtype.names)
tpi_I_acc_data = Dataset['low_end_imu_I']['accelerometer'][0,0]
print(tpi_I_acc_data.shape)
tpi_I_gyro_data = Dataset['low_end_imu_I']['gyroscope'][0,0].flatten()
print(tpi_I_gyro_data.shape)

# Unpack the nested arrays and extract the bias values
tpi_I_biases = {
    "bx": Dataset['low_end_imu_I']['biases'][0][0][0][0][0].item(),
    "by": Dataset['low_end_imu_I']['biases'][0][0][0][0][1].item(),
    "bz": Dataset['low_end_imu_I']['biases'][0][0][0][0][2].item(),
    "bw": Dataset['low_end_imu_I']['biases'][0][0][0][0][3].item()
}

print(tpi_I_biases.items())

In [ ]:
print(Dataset['low_end_imu_ID'].dtype.names)
tpi_ID_acc_data = Dataset['low_end_imu_ID']['accelerometer'][0,0]
print(tpi_ID_acc_data.shape)
tpi_ID_gyro_data = Dataset['low_end_imu_ID']['gyroscope'][0,0].flatten()
print(tpi_ID_gyro_data.shape)

# Unpack the nested arrays and extract the bias values
tpi_ID_biases = {
    "bx": Dataset['low_end_imu_ID']['biases'][0][0][0][0][0].item(),
    "by": Dataset['low_end_imu_ID']['biases'][0][0][0][0][1].item(),
    "bz": Dataset['low_end_imu_ID']['biases'][0][0][0][0][2].item(),
    "bw": Dataset['low_end_imu_ID']['biases'][0][0][0][0][3].item()
}

print(tpi_ID_biases.items())

In [6]:
odom_speed = Dataset['odometer']['speed'][0,0].flatten()
print(odom_speed.shape)

(2496,)


In [7]:
ref_data = (
    Dataset['reference_ins']['altitude'][0, 0].flatten(),  
    Dataset['reference_ins']['latitude'][0, 0].flatten(),  
    Dataset['reference_ins']['longitude'][0, 0].flatten(),  
    Dataset['reference_ins']['velocity_east'][0, 0].flatten(),  
    Dataset['reference_ins']['velocity_north'][0, 0].flatten(),  
    Dataset['reference_ins']['velocity_up'][0, 0].flatten(),  
    Dataset['reference_ins']['azimuth'][0, 0].flatten(),  
    Dataset['reference_ins']['pitch'][0, 0].flatten(),  
    Dataset['reference_ins']['roll'][0, 0].flatten()  
)

# Unpacking
ref_altitude, ref_latitude, ref_longitude, ref_ve, ref_vn, ref_vu, ref_az, ref_pitch, ref_roll = ref_data

## Mechanization Loop

In [8]:
num_epochs = len(time_grid.flatten())
num_epochs

2496

In [9]:
# Defining some constants
a = 6378137 # Semi-major axis (meters)
ecc = 0.08181919 # Eccentricity
omega_e = 7.292115e-5 # Earth’s rotation rate (rad/s)

In [10]:
def dead_reckoning(num_epochs, ref_data, acc_data, gyro_data, biases, odom_speed, omega_e, a, ecc, c, stop_mechanization):
    """Main function to perform dead reckoning."""
    # Initialize arrays
    alt_est, lat_est, long_est, ve_est, vn_est, vu_est, yaw_est, pitch_est, roll_est = initialize_arrays(num_epochs)
    filtered_speed = np.zeros(num_epochs)
    
    # Set initial values
    initialize_values(alt_est, lat_est, long_est, ve_est, vn_est, vu_est, yaw_est, pitch_est, roll_est, ref_data)
    filtered_speed[0] = odom_speed[0]
    bc, ac = initialize_butterworth_filter(cutoff=0.2, fs=1.0, order=2)
    
    # Loop through epochs
    for e in range(1, num_epochs):
        fx, fy, fz, omega_z, v_od = extract_sensor_data(e, acc_data, gyro_data, odom_speed, biases, c)

        if ((v_od == 0) and stop_mechanization):

            alt_est[e] = alt_est[e-1]
            lat_est[e] = lat_est[e-1]
            long_est[e] = long_est[e-1]
            ve_est[e] = 0
            vn_est[e] = 0
            vu_est[e] = 0
            yaw_est[e] = yaw_est[e-1]
            pitch_est[e] = pitch_est[e-1]
            roll_est[e] = roll_est[e-1]

            continue 
        
        compute_attitude(e, fx, fy, fz, omega_z, v_od, odom_speed, roll_est, pitch_est)
        #compute_attitude_ema(e, fx, fy, fz, omega_z, v_od, odom_speed, filtered_speed, roll_est, pitch_est, alpha=0.4, beta=0.1)
        #compute_attitude_bw(e, fx, fy, fz, omega_z, v_od, odom_speed, filtered_speed, roll_est, pitch_est, bc, ac, beta=0.1)
        compute_yaw(e, yaw_est, pitch_est, roll_est, lat_est, ve_est, alt_est, omega_z, omega_e, a, ecc)
        compute_velocity(e, yaw_est, pitch_est, v_od, ve_est, vn_est, vu_est)
        compute_position(e, lat_est, long_est, alt_est, ve_est, vn_est, vu_est, a, ecc)
    
    return alt_est, lat_est, long_est, ve_est, vn_est, vu_est, yaw_est, pitch_est, roll_est

In [11]:
alt_est, lat_est, long_est, ve_est, vn_est, vu_est, yaw_est, pitch_est, roll_est = dead_reckoning(num_epochs, 
                                                                                                  ref_data, novatel_acc_data, 
                                                                                                  novatel_gyro_data, novatel_biases, odom_speed,
                                                                                                    omega_e, a, ecc, -1,
                                                                                                    stop_mechanization=False)

In [12]:
novatel_est_data = (
    alt_est,
    lat_est,
    long_est,
    ve_est,
    vn_est,
    vu_est,
    yaw_est,
    pitch_est,
    roll_est
)

In [13]:
alt_est, lat_est, long_est, ve_est, vn_est, vu_est, yaw_est, pitch_est, roll_est = dead_reckoning(num_epochs, 
                                                                                                  ref_data, tpi_I_acc_data, 
                                                                                                  tpi_I_gyro_data, tpi_I_biases, odom_speed,
                                                                                                    omega_e, a, ecc, 1,
                                                                                                    stop_mechanization=True)

NameError: name 'tpi_I_acc_data' is not defined

In [ ]:
tpi_I_est_data = (
    alt_est,
    lat_est,
    long_est,
    ve_est,
    vn_est,
    vu_est,
    yaw_est,
    pitch_est,
    roll_est
)

In [ ]:
alt_est, lat_est, long_est, ve_est, vn_est, vu_est, yaw_est, pitch_est, roll_est = dead_reckoning(num_epochs, 
                                                                                                  ref_data, tpi_ID_acc_data, 
                                                                                                  tpi_ID_gyro_data, tpi_ID_biases, odom_speed,
                                                                                                    omega_e, a, ecc, 1,
                                                                                                    stop_mechanization=True)

In [ ]:
tpi_ID_est_data = (
    alt_est,
    lat_est,
    long_est,
    ve_est,
    vn_est,
    vu_est,
    yaw_est,
    pitch_est,
    roll_est
)

## Error Calculation

In [ ]:
# Compute total distance traveled
total_distance_traveled = calculate_total_distance_traveled(ref_data, a, ecc)

In [13]:
# Compute position errors
novatel_position_errors = calculate_position_errors(ref_data, novatel_est_data, a, ecc)

# Calculate maximum position error
novatel_max_error = calculate_max_position_error(novatel_position_errors)
print('Max: ', novatel_max_error.items())

# Calculate RMS position error
novatel_rms_error = calculate_rms_position_error(novatel_position_errors)
print('RMS: ', novatel_rms_error.items())

#print(total_distance_traveled.items())

# Calculate error percentage
#novatel_per_error = calculate_error_percentage(novatel_rms_error, total_distance_traveled)
#print(novatel_per_error.items())

Max:  dict_items([('east', 35.054953094156225), ('north', 52.804886708858575), ('2d', 54.80747917413511), ('altitude', 159.54281391398843), ('3d', 163.83086407490788)])
RMS:  dict_items([('east', 13.661642351133887), ('north', 27.42286835916126), ('2d', 30.637463680503696), ('altitude', 101.46713122682515), ('3d', 105.99166429571684)])


In [14]:
velocity_errors = calculate_velocity_errors(ref_data, novatel_est_data)

# Calculate STD velocity error
vel_std_error = calculate_std_velocity_error(velocity_errors)
print("Velocity Error STD: ", {k: round(v, 3) for k, v in vel_std_error.items()})

# Calculate RMS velocity error
vel_rms_error = calculate_rms_velocity_error(velocity_errors)
print("Velocity Error RMS: ", {k: round(v, 3) for k, v in vel_rms_error.items()})

# Calculate maximum velocity error
vel_max_error = calculate_max_velocity_error(velocity_errors)
print("Velocity Error Max: ", {k: round(v, 3) for k, v in vel_max_error.items()})

Velocity Error STD:  {'east': 0.326, 'north': 0.31, 'up': 0.36}
Velocity Error RMS:  {'east': 0.326, 'north': 0.31, 'up': 0.366}
Velocity Error Max:  {'east': 1.839, 'north': 1.782, 'up': 2.346}


In [ ]:
# Compute position errors
tpi_I_position_errors = calculate_position_errors(ref_data, tpi_I_est_data, a, ecc)

# Calculate maximum position error
tpi_I_max_error = calculate_max_position_error(tpi_I_position_errors)
print(tpi_I_max_error.items())

# Calculate RMS position error
tpi_I_rms_error = calculate_rms_position_error(tpi_I_position_errors)
print(tpi_I_rms_error.items())

print(total_distance_traveled.items())

# Calculate error percentage
tpi_I_per_error = calculate_error_percentage(tpi_I_rms_error, total_distance_traveled)
print(tpi_I_per_error.items())

In [ ]:
# Compute position errors
tpi_ID_position_errors = calculate_position_errors(ref_data, tpi_ID_est_data, a, ecc)

# Calculate maximum position error
tpi_ID_max_error = calculate_max_position_error(tpi_ID_position_errors)
print(tpi_ID_max_error.items())

# Calculate RMS position error
tpi_ID_rms_error = calculate_rms_position_error(tpi_ID_position_errors)
print(tpi_ID_rms_error.items())

print(total_distance_traveled.items())

# Calculate error percentage
tpi_ID_per_error = calculate_error_percentage(tpi_ID_rms_error, total_distance_traveled)
print(tpi_ID_per_error.items())

## Plotting

### Position

In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, np.degrees(novatel_est_data[1]), label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, np.degrees(tpi_I_est_data[1]), label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, np.degrees(ref_latitude), label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Angle (degrees)")
plt.title("Latitude Reference & Estimates Over Time")

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, np.degrees(novatel_est_data[2]), label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, np.degrees(tpi_I_est_data[2]), label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, np.degrees(ref_longitude), label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Angle (degrees)")
plt.title("Longitude Reference & Estimates Over Time")

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

# Plot reference trajectory
plt.plot(np.degrees(ref_longitude), np.degrees(ref_latitude), '-', color='#1f77b4', label="Reference")  # Blue

# Plot high-end estimate trajectory
plt.plot(np.degrees(novatel_est_data[2]), np.degrees(novatel_est_data[1]), '--', color='#fc8d62', label="High-end Estimate")  # Orange

# Plot low-end estimate trajectory
plt.plot(np.degrees(tpi_I_est_data[2]), np.degrees(tpi_I_est_data[1]), '--', color='#66c2a5', label="Low-end Estimate")  # Teal

# Labels and title
plt.xlabel("Longitude (Degrees)", fontsize=14, color='blue')
plt.ylabel("Latitude (Degrees)", fontsize=14, color='blue')
plt.title("Estimated vs Reference Trajectory", fontsize=16)

# Add a legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
# Create subplots with 1 row and 3 columns
fig, axs = plt.subplots(1, 3, figsize=(22, 5))

# Latitude Plot
axs[0].plot(time_grid, np.degrees(novatel_est_data[1]), label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[0].plot(time_grid, np.degrees(tpi_I_est_data[1]), label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[0].plot(time_grid, np.degrees(ref_latitude), label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Angle (degrees)")
axs[0].set_title("Latitude")
axs[0].grid(True)
axs[0].legend()

# Longitude Plot
axs[1].plot(time_grid, np.degrees(novatel_est_data[2]), label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[1].plot(time_grid, np.degrees(tpi_I_est_data[2]), label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[1].plot(time_grid, np.degrees(ref_longitude), label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Angle (degrees)")
axs[1].set_title("Longitude")
axs[1].grid(True)
axs[1].legend()

# Trajectory Plot
axs[2].plot(np.degrees(ref_longitude), np.degrees(ref_latitude), '-', color='#1f77b4', label="Reference")  # Blue
axs[2].plot(np.degrees(novatel_est_data[2]), np.degrees(novatel_est_data[1]), '--', color='#fc8d62', label="High-end Estimate")  # Orange
axs[2].plot(np.degrees(tpi_I_est_data[2]), np.degrees(tpi_I_est_data[1]), '--', color='#66c2a5', label="Low-end Estimate")  # Teal
axs[2].set_xlabel("Longitude (Degrees)")
axs[2].set_ylabel("Latitude (Degrees)")
axs[2].set_title("Trajectory")
axs[2].grid(True)
axs[2].legend()

# Adjust layout for better spacing
plt.tight_layout()

# Show the combined plot
plt.show()


In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, novatel_est_data[0], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, tpi_I_est_data[0], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, ref_altitude, label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Altitude (meters)")
plt.title("Altitude Reference & Estimates Over Time")
plt.ylim(0,100)

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

#plt.savefig("alt_filt.pdf", format="pdf")

# Show the plot
plt.show()


### Velocity

In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, novatel_est_data[3], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, tpi_I_est_data[3], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, ref_ve, label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Velocity (m/s)")
plt.title("East Velocity Reference & Estimates Over Time")

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, novatel_est_data[4], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, tpi_I_est_data[4], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, ref_vn, label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Velocity (m/s)")
plt.title("North Velocity Reference & Estimates Over Time")

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, novatel_est_data[5], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, tpi_I_est_data[5], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, ref_vu, label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Velocity (m/s)")
plt.title("Up Velocity Reference & Estimates Over Time")

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
# Create subplots with 1 row and 3 columns
fig, axs = plt.subplots(1, 3, figsize=(22, 5))

# East Velocity Plot
axs[0].plot(time_grid, novatel_est_data[3], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[0].plot(time_grid, tpi_I_est_data[3], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[0].plot(time_grid, ref_ve, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Velocity (m/s)")
axs[0].set_title("East Velocity")
axs[0].grid(True)
axs[0].legend()

# North Velocity Plot
axs[1].plot(time_grid, novatel_est_data[4], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[1].plot(time_grid, tpi_I_est_data[4], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[1].plot(time_grid, ref_vn, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Velocity (m/s)")
axs[1].set_title("North Velocity")
axs[1].grid(True)
axs[1].legend()

# Up Velocity Plot
axs[2].plot(time_grid, novatel_est_data[5], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[2].plot(time_grid, tpi_I_est_data[5], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[2].plot(time_grid, ref_vu, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[2].set_xlabel("Time")
axs[2].set_ylabel("Velocity (m/s)")
axs[2].set_title("Up Velocity")
axs[2].grid(True)
axs[2].legend()

# Adjust layout for better spacing
plt.tight_layout()

# Show the combined plot
plt.show()


### Attitude

In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, novatel_est_data[6], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, tpi_I_est_data[6], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, -1 * ref_az % (2 * np.pi), label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Angle (radians)")
plt.title("Yaw Reference & Estimates Over Time")

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, novatel_est_data[7], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, tpi_I_est_data[7], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, ref_pitch, label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Angle (radians)")
plt.title("Pitch Reference & Estimates Over Time")

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
# Plot the data
plt.figure(figsize=(10, 5))

plt.plot(time_grid, novatel_est_data[8], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
plt.plot(time_grid, tpi_I_est_data[8], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
plt.plot(time_grid, ref_roll, label="Reference", linestyle='-', color='#1f77b4')  # Blue

# Labels and title
plt.xlabel("Time")
plt.ylabel("Angle (radians)")
plt.title("Roll Reference & Estimates Over Time")

# Legend
plt.legend()

# Grid for better visualization
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
# Create subplots with 1 row and 3 columns
fig, axs = plt.subplots(1, 3, figsize=(22, 5))

# Yaw Plot
axs[0].plot(time_grid, novatel_est_data[6], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[0].plot(time_grid, tpi_I_est_data[6], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[0].plot(time_grid, -1 * ref_az % (2 * np.pi), label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Angle (radians)")
axs[0].set_title("Yaw")
axs[0].grid(True)
axs[0].legend()

# Pitch Plot
axs[1].plot(time_grid, novatel_est_data[7], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[1].plot(time_grid, tpi_I_est_data[7], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[1].plot(time_grid, ref_pitch, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Angle (radians)")
axs[1].set_title("Pitch")
axs[1].grid(True)
axs[1].legend()

# Roll Plot
axs[2].plot(time_grid, novatel_est_data[8], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[2].plot(time_grid, tpi_I_est_data[8], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[2].plot(time_grid, ref_roll, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[2].set_xlabel("Time")
axs[2].set_ylabel("Angle (radians)")
axs[2].set_title("Roll")
axs[2].grid(True)
axs[2].legend()

# Adjust layout for better spacing
plt.tight_layout()

# Show the combined plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a 3x3 grid of subplots
fig, axs = plt.subplots(3, 3, figsize=(18, 12))

# --- Row 1: Latitude, Longitude, Trajectory ---
# Latitude Plot
axs[0, 0].plot(time_grid, np.degrees(novatel_est_data[1]), label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[0, 0].plot(time_grid, np.degrees(tpi_I_est_data[1]), label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[0, 0].plot(time_grid, np.degrees(ref_latitude), label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[0, 0].set_title("Latitude")
axs[0, 0].set_xlabel("Time")
axs[0, 0].set_ylabel("Angle (degrees)")
axs[0, 0].legend()
axs[0, 0].grid(True)

# Longitude Plot
axs[0, 1].plot(time_grid, np.degrees(novatel_est_data[2]), label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[0, 1].plot(time_grid, np.degrees(tpi_I_est_data[2]), label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[0, 1].plot(time_grid, np.degrees(ref_longitude), label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[0, 1].set_title("Longitude")
axs[0, 1].set_xlabel("Time")
axs[0, 1].set_ylabel("Angle (degrees)")
axs[0, 1].legend()
axs[0, 1].grid(True)

# Trajectory Plot
axs[0, 2].plot(np.degrees(ref_longitude), np.degrees(ref_latitude), '-', color='#1f77b4', label="Reference")  # Blue
axs[0, 2].plot(np.degrees(novatel_est_data[2]), np.degrees(novatel_est_data[1]), '--', color='#fc8d62', label="High-end Estimate")  # Orange
axs[0, 2].plot(np.degrees(tpi_I_est_data[2]), np.degrees(tpi_I_est_data[1]), '--', color='#66c2a5', label="Low-end Estimate")  # Teal
axs[0, 2].set_title("Trajectory")
axs[0, 2].set_xlabel("Longitude (degrees)")
axs[0, 2].set_ylabel("Latitude (degrees)")
axs[0, 2].legend()
axs[0, 2].grid(True)

# --- Row 2: East, North, Up Velocity ---
# East Velocity Plot
axs[1, 0].plot(time_grid, novatel_est_data[3], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[1, 0].plot(time_grid, tpi_I_est_data[3], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[1, 0].plot(time_grid, ref_ve, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[1, 0].set_title("East Velocity")
axs[1, 0].set_xlabel("Time")
axs[1, 0].set_ylabel("Velocity (m/s)")
axs[1, 0].legend()
axs[1, 0].grid(True)

# North Velocity Plot
axs[1, 1].plot(time_grid, novatel_est_data[4], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[1, 1].plot(time_grid, tpi_I_est_data[4], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[1, 1].plot(time_grid, ref_vn, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[1, 1].set_title("North Velocity")
axs[1, 1].set_xlabel("Time")
axs[1, 1].set_ylabel("Velocity (m/s)")
axs[1, 1].legend()
axs[1, 1].grid(True)

# Up Velocity Plot
axs[1, 2].plot(time_grid, novatel_est_data[5], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[1, 2].plot(time_grid, tpi_I_est_data[5], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[1, 2].plot(time_grid, ref_vu, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[1, 2].set_title("Up Velocity")
axs[1, 2].set_xlabel("Time")
axs[1, 2].set_ylabel("Velocity (m/s)")
axs[1, 2].legend()
axs[1, 2].grid(True)

# --- Row 3: Yaw, Pitch, Roll ---
# Yaw Plot
axs[2, 0].plot(time_grid, novatel_est_data[6], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[2, 0].plot(time_grid, tpi_I_est_data[6], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[2, 0].plot(time_grid, -1 * ref_az % (2 * np.pi), label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[2, 0].set_title("Yaw")
axs[2, 0].set_xlabel("Time")
axs[2, 0].set_ylabel("Angle (radians)")
axs[2, 0].legend()
axs[2, 0].grid(True)

# Pitch Plot
axs[2, 1].plot(time_grid, novatel_est_data[7], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[2, 1].plot(time_grid, tpi_I_est_data[7], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[2, 1].plot(time_grid, ref_pitch, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[2, 1].set_title("Pitch")
axs[2, 1].set_xlabel("Time")
axs[2, 1].set_ylabel("Angle (radians)")
axs[2, 1].legend()
axs[2, 1].grid(True)

# Roll Plot
axs[2, 2].plot(time_grid, novatel_est_data[8], label="High-end Estimate", linestyle='-', color='#fc8d62')  # Orange
axs[2, 2].plot(time_grid, tpi_I_est_data[8], label="Low-end Estimate", linestyle='-', color='#66c2a5')  # Teal
axs[2, 2].plot(time_grid, ref_roll, label="Reference", linestyle='-', color='#1f77b4')  # Blue
axs[2, 2].set_title("Roll")
axs[2, 2].set_xlabel("Time")
axs[2, 2].set_ylabel("Angle (radians)")
axs[2, 2].legend()
axs[2, 2].grid(True)

# Adjust layout to ensure no overlap
plt.tight_layout()

# Save as PDF (optional)
#plt.savefig("combined_plot.pdf", format="pdf")

# Show the combined plot
plt.show()
